# 1. Import Libraries

In [1]:
import numpy as np
import tensorflow as tf
import itertools
import sys
import os
import gc
import json
import optuna

c:\Users\USER\Documents\Folder-Proposal\Programs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Data & Model Paths

## 1. Folders

In [2]:
scripts_folder = os.path.abspath(os.path.join('..', 'Scripts'))
data_folder = os.path.abspath(os.path.join('..', 'Data', 'TrainTest'))
hyper_parameter_folder = os.path.abspath(os.path.join('..', 'Data', 'HyperParameters'))

## 2. System paths

In [3]:
sys.path.append(scripts_folder)

# 3. Import Data & Models

## 1. Import Models

In [4]:
from model import Encoder, Decoder, VAE, RSVD

## 2. Import Data

In [5]:
train_data_path = os.path.join(data_folder, 'train_data.npy')

# Import training data
try:
    train_data = np.load(train_data_path).astype(np.float32)
    train_data_tf = tf.constant(train_data, dtype=tf.float32) # Mencegah tensorflow duplikasi data tiap iterasi (memory leak)
    # Mengambil jumlah item 
    num_items = train_data.shape[1]
    print(f"Data latih berhasil dimuat.")
    print(f"   Dimensi data: {train_data.shape} (Pengguna x Film)")
except FileNotFoundError:
    print("Error: File 'train_data.npy' tidak ditemukan. Pastikan Anda sudah menjalankan preprocessing.")

Data latih berhasil dimuat.
   Dimensi data: (943, 1682) (Pengguna x Film)


# 4. Hyperparameter Tuning

## 1. VAE

### 1. Search Space

In [ ]:
# Ruang pencarian hyperparameter VAE untuk Optuna
# Optuna akan menjelajahi ruang ini secara cerdas menggunakan algoritma TPE
vae_search_space = {
    'latent_dim'   : [20, 50, 100, 200],
    'learning_rate': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
    'batch_size'   : [64, 128, 256],
    'dropout_rate' : [0.1, 0.2, 0.3, 0.4],
    'hidden_dims'  : [
        [512, 256],
        [512, 256, 128],
        [1024, 512],
        [1024, 512, 256],
    ],
    'beta'         : [0.01, 0.05, 0.1, 0.3, 0.5, 0.8, 1.0],
}

# Jumlah trial yang akan dijalankan Optuna
# Optuna memilih kombinasi secara cerdas berdasarkan hasil trial sebelumnya (TPE)
N_TRIALS_VAE = 200

print(f"[INFO] Search space VAE:")
for key, values in vae_search_space.items():
    print(f"       {key:<15}: {values}")
print(f"\n[INFO] Jumlah trial yang akan dijalankan: {N_TRIALS_VAE}")

[INFO] Search space VAE:
       latent_dim     : [20, 50, 100, 200]
       learning_rate  : [0.0001, 0.0005, 0.001, 0.005, 0.01]
       batch_size     : [64, 128, 256]
       dropout_rate   : [0.1, 0.2, 0.3, 0.4]
       hidden_dims    : [[512, 256], [512, 256, 128], [1024, 512], [1024, 512, 256]]
       beta           : [0.01, 0.05, 0.1, 0.3, 0.5, 0.8, 1.0]

[INFO] Jumlah trial yang akan dijalankan: 200
[INFO] Total kombinasi jika grid search: 3840 kombinasi


### 2. Optuna Objective Function

In [7]:
# Nonaktifkan logging bawaan Optuna agar output lebih bersih
optuna.logging.set_verbosity(optuna.logging.WARNING)

def vae_objective(trial):
    """
    Fungsi objektif yang dipanggil Optuna untuk setiap trial.
    Optuna menggunakan hasil fungsi ini untuk menentukan kombinasi
    parameter berikutnya yang paling menjanjikan (algoritma TPE).

    Args:
        trial : objek Optuna yang menyediakan metode suggest_*
                untuk memilih nilai hyperparameter

    Returns:
        reconstruction_loss : float, nilai yang diminimalkan Optuna.
                              Dipakai reconstruction_loss (bukan total loss)
                              agar hasil tidak bias terhadap nilai beta kecil.
    """
    # Optuna memilih nilai untuk setiap hyperparameter dari search space
    latent_dim    = trial.suggest_categorical('latent_dim',    vae_search_space['latent_dim'])
    learning_rate = trial.suggest_categorical('learning_rate', vae_search_space['learning_rate'])
    batch_size    = trial.suggest_categorical('batch_size',    vae_search_space['batch_size'])
    dropout_rate  = trial.suggest_categorical('dropout_rate',  vae_search_space['dropout_rate'])
    beta          = trial.suggest_categorical('beta',          vae_search_space['beta'])

    # hidden_dims dikonversi ke string karena suggest_categorical hanya menerima
    # tipe primitif (str, int, float), lalu dikonversi kembali ke list
    hidden_dims_str = trial.suggest_categorical('hidden_dims', [str(h) for h in vae_search_space['hidden_dims']])
    hidden_dims     = eval(hidden_dims_str)

    # Bersihkan sesi TensorFlow sebelum membuat model baru
    tf.keras.backend.clear_session()
    tf.compat.v1.reset_default_graph()

    # Bangun arsitektur VAE dengan hyperparameter yang dipilih Optuna
    encoder = Encoder(hidden_dims=hidden_dims, latent_dim=latent_dim, dropout_rate=dropout_rate)
    decoder = Decoder(hidden_dims=hidden_dims[::-1], output_dim=num_items)
    vae     = VAE(encoder, decoder, beta=beta)

    dummy_output = vae(train_data_tf[:1])

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    vae.compile(optimizer=optimizer, run_eagerly=True)

    # Latih model (30 epoch agar konvergensi lebih terbaca)
    history = vae.fit(
        train_data_tf, train_data_tf,
        epochs     = 30,
        batch_size = batch_size,
        verbose    = 0
    )

    # Ambil reconstruction_loss sebagai kriteria seleksi
    reconstruction_loss = history.history['reconstruction_loss'][-1]

    # Bersihkan memori sebelum kembali ke Optuna
    del encoder, decoder, vae, history, optimizer, dummy_output
    gc.collect()
    tf.keras.backend.clear_session()

    return reconstruction_loss

### 3. Optuna Study (Resume-able)

In [8]:
best_vae_path     = os.path.join(hyper_parameter_folder, 'best_vae_weights.weights.h5')
vae_progress_file = os.path.join(hyper_parameter_folder, 'tuning_progress_vae.json')
vae_study_db      = os.path.join(hyper_parameter_folder, 'optuna_vae_study.db')

# Buat atau load kembali study yang sudah ada
# storage SQLite memungkinkan study dilanjutkan jika proses Jupyter terputus
study_vae = optuna.create_study(
    study_name     = 'vae_tuning',
    direction      = 'minimize',
    storage        = f'sqlite:///{vae_study_db}',
    load_if_exists = True    # otomatis resume jika study sudah ada
)

# Hitung berapa trial yang sudah selesai
n_completed = len([t for t in study_vae.trials if t.state == optuna.trial.TrialState.COMPLETE])
n_remaining = max(0, N_TRIALS_VAE - n_completed)

print(f"[INFO] Trial sudah selesai : {n_completed}")
print(f"[INFO] Trial tersisa       : {n_remaining}")

if n_completed > 0:
    print(f"[INFO] Best reconstruction loss sejauh ini: {study_vae.best_value:.6f}")
    print(f"[INFO] Best params sejauh ini             : {study_vae.best_params}")

[INFO] Trial sudah selesai : 0
[INFO] Trial tersisa       : 200


### 4. Run Tuning

In [9]:
# Callback untuk mencetak progress setiap trial selesai
def print_trial_callback(study, trial):
    print(f"  Trial {trial.number+1:>3} | Recon Loss: {trial.value:.6f} | Params: {trial.params}")
    if study.best_trial.number == trial.number:
        print(f"         [NEW BEST] Reconstruction Loss turun ke {trial.value:.6f}")

if n_remaining > 0:
    print(f"[INFO] Memulai {n_remaining} trial Optuna (TPE)...\n")
    study_vae.optimize(
        vae_objective,
        n_trials       = n_remaining,
        callbacks      = [print_trial_callback],
        gc_after_trial = True    # bersihkan memori setelah setiap trial
    )
else:
    print("[INFO] Semua trial sudah selesai. Tidak ada yang perlu dijalankan.")

print(f"\n[SELESAI] Best Reconstruction Loss : {study_vae.best_value:.6f}")
print(f"[SELESAI] Best Params               : {study_vae.best_params}")

[INFO] Memulai 200 trial Optuna (TPE)...


  Trial   1 | Recon Loss: 61.073570 | Params: {'latent_dim': 50, 'learning_rate': 0.01, 'batch_size': 128, 'dropout_rate': 0.2, 'beta': 0.8, 'hidden_dims': '[512, 256, 128]'}
         [NEW BEST] Reconstruction Loss turun ke 61.073570
  Trial   2 | Recon Loss: 51.393631 | Params: {'latent_dim': 50, 'learning_rate': 0.001, 'batch_size': 128, 'dropout_rate': 0.1, 'beta': 0.5, 'hidden_dims': '[512, 256]'}
         [NEW BEST] Reconstruction Loss turun ke 51.393631
  Trial   3 | Recon Loss: 46.194454 | Params: {'latent_dim': 50, 'learning_rate': 0.0005, 'batch_size': 128, 'dropout_rate': 0.4, 'beta': 0.1, 'hidden_dims': '[512, 256]'}
         [NEW BEST] Reconstruction Loss turun ke 46.194454
  Trial   4 | Recon Loss: 54.856007 | Params: {'latent_dim': 200, 'learning_rate': 0.005, 'batch_size': 64, 'dropout_rate': 0.3, 'beta': 0.3, 'hidden_dims': '[1024, 512, 256]'}
  Trial   5 | Recon Loss: 50.239262 | Params: {'latent_dim': 200, 'learning_rate': 0.

### 5. Simpan Hasil Terbaik

In [10]:
# Rekonstruksi hidden_dims dari string kembali ke list
best_params_raw = study_vae.best_params.copy()
best_params_raw['hidden_dims'] = eval(best_params_raw['hidden_dims'])

# Simpan ke format JSON yang sama dengan sebelumnya
# sehingga Training.ipynb, CrossValidation.ipynb, dan Testing.ipynb
# tidak perlu diubah sama sekali
with open(vae_progress_file, 'w') as f:
    json.dump({
        'best_loss'  : study_vae.best_value,
        'best_params': best_params_raw,
    }, f, indent=4)

print(f"[SUCCESS] Parameter terbaik disimpan ke: {vae_progress_file}")
print(f"[INFO]    Isi: {best_params_raw}")

[SUCCESS] Parameter terbaik disimpan ke: c:\Users\USER\Documents\Folder-Proposal\Programs\Data\HyperParameters\tuning_progress_vae.json
[INFO]    Isi: {'latent_dim': 100, 'learning_rate': 0.001, 'batch_size': 64, 'dropout_rate': 0.1, 'beta': 0.01, 'hidden_dims': [1024, 512]}


### 6. Simpan Bobot Model Terbaik

In [11]:
# Re-train model dengan parameter terbaik untuk menyimpan bobot
# Optuna tidak menyimpan bobot model selama proses search, hanya nilai loss-nya
print("[INFO] Melatih ulang model terbaik untuk menyimpan bobot...\n")

best_hidden_dims = best_params_raw['hidden_dims']

tf.keras.backend.clear_session()

encoder_best = Encoder(
    hidden_dims  = best_hidden_dims,
    latent_dim   = best_params_raw['latent_dim'],
    dropout_rate = best_params_raw['dropout_rate']
)
decoder_best = Decoder(
    hidden_dims = best_hidden_dims[::-1],
    output_dim  = num_items
)
vae_best = VAE(encoder_best, decoder_best, beta=best_params_raw['beta'])

dummy_output = vae_best(train_data_tf[:1])

optimizer_best = tf.keras.optimizers.Adam(learning_rate=best_params_raw['learning_rate'])
vae_best.compile(optimizer=optimizer_best, run_eagerly=True)

vae_best.fit(
    train_data_tf, train_data_tf,
    epochs     = 30,
    batch_size = best_params_raw['batch_size'],
    verbose    = 1
)

# Simpan bobot ke file yang sama dengan sebelumnya
vae_best.save_weights(best_vae_path)
print(f"\n[SUCCESS] Bobot model terbaik disimpan ke: {best_vae_path}")

# Bersihkan memori
del encoder_best, decoder_best, vae_best, optimizer_best, dummy_output
gc.collect()
tf.keras.backend.clear_session()

[INFO] Melatih ulang model terbaik untuk menyimpan bobot...

Epoch 1/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - kl_loss: 8.8047 - loss: 90.9061 - reconstruction_loss: 90.8181   
Epoch 2/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - kl_loss: 0.5109 - loss: 71.7947 - reconstruction_loss: 71.7896
Epoch 3/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - kl_loss: 1.3114 - loss: 69.8922 - reconstruction_loss: 69.8791
Epoch 4/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - kl_loss: 4.4585 - loss: 67.0269 - reconstruction_loss: 66.9823
Epoch 5/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - kl_loss: 9.9612 - loss: 64.4950 - reconstruction_loss: 64.3954
Epoch 6/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - kl_loss: 18.0248 - loss: 63.0361 - reconstruction_loss: 62.8558
Epoch 7/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - kl_loss: 30.7325 - loss: 60.1552 - reconstruction_loss: 59.8479
Epoch 8/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - kl_loss: 35.9066 - loss: 57.6883 - reconstruction_loss: 57.3293
Epoch

## 2. RSVD

### 1. Hyperparameter Search Space

In [12]:
# Ruang pencarian hyperparameter
rsvd_param_grid = {
    'n_factors': [10, 20, 50],              
    'learning_rate': [0.001, 0.005, 0.01, 0.02], 
    'lambda_reg': [0.001, 0.005, 0.01, 0.02]      
}

rsvd_keys, rsvd_values = zip(*rsvd_param_grid.items())
rsvd_combinations = [dict(zip(rsvd_keys, v)) for v in itertools.product(*rsvd_values)]

best_rsvd_error = float('inf')
best_rsvd_params = None
best_rsvd_model = None

### 2. Grid Search

#### 1. Load Progress

In [13]:
rsvd_progress_file = os.path.join(hyper_parameter_folder, 'tuning_progress_rsvd.json')

if os.path.exists(rsvd_progress_file):
    with open(rsvd_progress_file, 'r') as f:
        progress_data = json.load(f)
    
    best_rsvd_error = progress_data['best_error']
    best_rsvd_params = progress_data['best_params']
    completed_indices = progress_data['completed_indices']
    
    print(f"\n[INFO] Melanjutkan sesi RSVD sebelumnya...")
    print(f"[INFO] {len(completed_indices)} kombinasi sudah dievaluasi.")
    print(f"[INFO] MSE terbaik saat ini: {best_rsvd_error:.4f}")
else:
    best_rsvd_error = float('inf')
    best_rsvd_params = None
    completed_indices = []


[INFO] Melanjutkan sesi RSVD sebelumnya...
[INFO] 48 kombinasi sudah dievaluasi.
[INFO] MSE terbaik saat ini: 0.2801


#### 2. Looping

In [14]:
for i, params in enumerate(rsvd_combinations):
    # Lewati iterasi jika kombinasi ini sudah pernah dikerjakan
    if i in completed_indices:
        continue
    
    print(f"\n[{i+1}/{len(rsvd_combinations)}] Menguji RSVD | Params: {params}")
    
    # Inisialisasi model RSVD dengan parameter saat ini
    rsvd = RSVD(
        n_factors=params['n_factors'], 
        learning_rate=params['learning_rate'], 
        lambda_reg=params['lambda_reg'], 
        epochs=15  # Epoch kecil untuk tuning
    )
    
    # Latih RSVD menggunakan data training asli
    rsvd.fit(train_data)
    
    total_loss_error = rsvd.loss_history[-1]
    
    print(f"   -> MSE : {total_loss_error:.4f}")
    
    # Cek apakah ini kombinasi terbaik
    if total_loss_error < best_rsvd_error:
        print(f"   🌟 [NEW BEST FOUND!] MSE turun ke {total_loss_error:.4f}. Menyimpan model...")
        best_rsvd_error = total_loss_error
        best_rsvd_params = params
        best_rsvd_model = rsvd

        # Menyimpan komponen RSVD jika mendapat skor terbaik
        np.save(os.path.join(hyper_parameter_folder, 'best_mu.npy'), np.array(rsvd.mu))
        np.save(os.path.join(hyper_parameter_folder, 'best_b_u.npy'), rsvd.b_u)
        np.save(os.path.join(hyper_parameter_folder, 'best_b_i.npy'), rsvd.b_i)
        np.save(os.path.join(hyper_parameter_folder, 'best_U.npy'), rsvd.U)
        np.save(os.path.join(hyper_parameter_folder, 'best_Sigma.npy'), rsvd.Sigma)
        np.save(os.path.join(hyper_parameter_folder, 'best_V.npy'), rsvd.V)

    completed_indices.append(i)
    with open(rsvd_progress_file, 'w') as f:
        json.dump({
            'best_error': best_rsvd_error,
            'best_params': best_rsvd_params,
            'completed_indices': completed_indices
        }, f)
        
    # Bersihkan variabel array raksasa dari memori untuk jaga-jaga
    del rsvd
    gc.collect()

print("\n==================================================")
print(f"[HASIL TERBAIK RSVD]")
print(f"Parameter Terbaik: {best_rsvd_params}")
print(f"MSE Terendah pada Ruang Laten: {best_rsvd_error:.4f}")
print("==================================================")


[HASIL TERBAIK RSVD]
Parameter Terbaik: {'n_factors': 10, 'learning_rate': 0.005, 'lambda_reg': 0.001}
MSE Terendah pada Ruang Laten: 0.2801
